In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

# Optimized cell 5
# 1. Filter & project lineitem on the GPU using string dates (avoids pd.Timestamp)
mask = (lineitem.L_SHIPDATE >= "1995-01-01") & (lineitem.L_SHIPDATE < "1997-01-01")
li = lineitem[mask][[
    "L_ORDERKEY", "L_SUPPKEY", "L_SHIPDATE", "L_EXTENDEDPRICE", "L_DISCOUNT"
]]
# extract year by slicing the first 4 characters instead of dt.year
li["L_YEAR"] = li.L_SHIPDATE.str.slice(start=0, stop=4).astype("int16")
# compute volume
li["VOLUME"] = li.L_EXTENDEDPRICE * (1 - li.L_DISCOUNT)
# keep only the needed columns
li = li[["L_ORDERKEY", "L_SUPPKEY", "L_YEAR", "VOLUME"]]

# 2. Pre-filter nation table once for both FRANCE and GERMANY
nations = nation[nation.N_NAME.isin(["FRANCE", "GERMANY"])][["N_NATIONKEY", "N_NAME"]]

# 3. Attach nation names to customer and supplier
cust = (
    customer
    .merge(nations, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner")[["C_CUSTKEY", "N_NAME"]]
    .rename(columns={"N_NAME": "CUST_NATION"})
)
supp = (
    supplier
    .merge(nations, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner")[["S_SUPPKEY", "N_NAME"]]
    .rename(columns={"N_NAME": "SUPP_NATION"})
)

# 4. Join lineitem → orders → customer → supplier, then filter the two nation pairs
df = (
    li
    .merge(orders[["O_ORDERKEY", "O_CUSTKEY"]], left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner")
    .merge(cust,   left_on="O_CUSTKEY", right_on="C_CUSTKEY", how="inner")
    .merge(supp,   left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner")[["SUPP_NATION", "CUST_NATION", "L_YEAR", "VOLUME"]]
)

df = df[
    ((df.SUPP_NATION == "FRANCE") & (df.CUST_NATION == "GERMANY"))
    | ((df.SUPP_NATION == "GERMANY") & (df.CUST_NATION == "FRANCE"))
]

# 5. Aggregate on GPU and rename
total = (
    df
    .groupby(["SUPP_NATION", "CUST_NATION", "L_YEAR"], as_index=False)["VOLUME"]
    .sum()
    .rename(columns={"VOLUME": "REVENUE"})
)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/rewritten/o4_mini_high/checkpoints/post_cell_5_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q07/opt_cell_exec_info_5_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [ ]:
opt_output = Out.get(4)